# QUERY 2
Nombre de banco, cuenta de origen y monto de la max. transacción USD de cada banco.

In [1]:
# ── Configuración ──────────────────────────────────────────────────────────

RUTA_DE_DATASETS             = "../../datasets"
# NOMBRE_ARCHIVO_TRANSACCIONES = "trans_sample.csv"
NOMBRE_ARCHIVO_TRANSACCIONES = "transacciones_sample_q4.csv"
NOMBRE_ARCHIVO_CUENTAS       = "LI-Small_accounts.csv"

# Archivos intermedios donde guardaremos la data YA normalizada
RUTA_TRANSACCIONES_NORM      = "transacciones_normalizadas.csv"
RUTA_CUENTAS_NORM            = "cuentas_normalizadas.csv"

RUTA_RESULTADO_QUERY2        = "q2_solucion.csv"

In [2]:
import pandas as pd

def normalizar_bank_id(serie):
    return serie.str.strip().str.lstrip("0").replace("", "0")

guardar_csv = lambda df, ruta: df.to_csv(ruta, index=False)

In [3]:
# ── PASO 1: Creación de archivos normalizados ─────────────────────────────
print("Generando archivos normalizados...")

# 1. Leer los datasets originales
transacciones = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_TRANSACCIONES}",
    dtype={"From Bank": "string", "Account": "string",
           "To Bank": "string", "Account.1": "string"}
)
cuentas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_CUENTAS}",
    dtype={"Bank ID": "string", "Account Number": "string"}
)

# 2. Aplicar normalización y crear las nuevas columnas
transacciones["From Bank Normalized"] = normalizar_bank_id(transacciones["From Bank"])
transacciones["To Bank Normalized"]   = normalizar_bank_id(transacciones["To Bank"])
cuentas["Bank ID Normalized"]         = normalizar_bank_id(cuentas["Bank ID"])

# 3. Guardar los archivos resultantes para usarlos en el siguiente paso
guardar_csv(transacciones, RUTA_TRANSACCIONES_NORM)
guardar_csv(cuentas, RUTA_CUENTAS_NORM)

print(f"  → Guardado en {RUTA_TRANSACCIONES_NORM}")
print(f"  → Guardado en {RUTA_CUENTAS_NORM}")

Generando archivos normalizados...
  → Guardado en transacciones_normalizadas.csv
  → Guardado en cuentas_normalizadas.csv


In [4]:
# ── PASO 2: Procesar la consulta desde los nuevos archivos ────────────────
print("Procesando la consulta desde archivos normalizados...")

# 1. Cargar la data desde los archivos que acabamos de crear
df_transacciones = pd.read_csv(
    RUTA_TRANSACCIONES_NORM,
    dtype={"From Bank Normalized": "string", "Payment Currency": "string"}
)
df_cuentas = pd.read_csv(
    RUTA_CUENTAS_NORM,
    dtype={"Bank ID Normalized": "string", "Bank ID": "string"}
)

# 2. Filtrar por moneda y limpiar montos
transacciones_usd = df_transacciones[df_transacciones["Payment Currency"] == "US Dollar"].copy()
transacciones_usd["Amount Paid"] = pd.to_numeric(transacciones_usd["Amount Paid"], errors="coerce")
transacciones_usd = transacciones_usd.dropna(subset=["Amount Paid"])

# 3. Preparar el dataframe de bancos usando el ID ya normalizado
bancos = df_cuentas[["Bank ID Normalized", "Bank ID", "Bank Name"]].rename(
    columns={"Bank ID Normalized": "From Bank Normalized"}
).drop_duplicates(subset=["From Bank Normalized"])

# 4. Unir transacciones con cuentas
transacciones_con_banco = transacciones_usd.merge(
    bancos, on="From Bank Normalized", how="inner"
)

# 5. Agrupar, buscar máximos y estructurar resultado final
idx_maximos = transacciones_con_banco.groupby("From Bank Normalized")["Amount Paid"].idxmax()
resultado_query2 = transacciones_con_banco.loc[
    idx_maximos, ["Bank ID", "Account", "Bank Name", "Amount Paid"]
].rename(columns={"Bank ID": "From Bank"}).sort_values(by="From Bank").reset_index(drop=True)

# 6. Guardar y mostrar el resultado
guardar_csv(resultado_query2, RUTA_RESULTADO_QUERY2)

print(f"Query finalizada. Resultado guardado en {RUTA_RESULTADO_QUERY2}")
print(resultado_query2.head())

Procesando la consulta desde archivos normalizados...
Query finalizada. Resultado guardado en q2_solucion.csv
  From Bank    Account               Bank Name   Amount Paid
0         0  801797710          Italy Bank #18  5.938537e+06
1         1  800908DF0  First Bank of Portland  5.280050e+08
2        10  80E6CF460         Canada Bank #18  8.879874e+04
3     10073  803D81830         China Bank #114  7.157727e+07
4     10154  80445C950         China Bank #448  5.578667e+04
